# Fase 5 · M05: MLP + EBM

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | M05 — MLP y Explainable Boosting Machine |

---

## 🎯 Qué hace

Entrena y evalúa dos modelos complementarios sobre el dataset D_strict
(19 features + target `abandono`):

- **MLPClassifier** — red neuronal multicapa (sklearn). Alta capacidad no lineal,
  poco interpretable. Se analiza su arquitectura (capas vs AUC).
- **EBM** — Explainable Boosting Machine (InterpretML). GAM + boosting: rendimiento
  comparable al MLP con interpretabilidad nativa por feature.

> **Nota:** Keras y PyTorch quedan fuera del alcance de este TFM. Para datos
> tabulares con 19 features, MLPClassifier ofrece el mismo poder expresivo sin
> la sobrecarga de frameworks de deep learning, y facilita la Fase 6 (SHAP).

## 📋 Requisitos

- `data/05_modelado/X_train_prep.parquet`, `X_test_prep.parquet`
- `data/05_modelado/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`
- Entorno: `tfm_abandono` (scikit-learn ≥1.3, interpret, optuna, imbalanced-learn, codecarbon)
- Instalar InterpretML si no está: `pip install interpret --break-system-packages`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/results/results_mlp_ebm.parquet` | Métricas 6 combinaciones |
| `data/05_modelado/results/results_mlp_ebm.xlsx` | Mismas métricas en Excel |
| `data/05_modelado/results/results_mlp_ebm.json` | Metadatos + hiperparámetros |
| `data/05_modelado/models/MLP__*.pkl` | 3 modelos MLP serializados |
| `data/05_modelado/models/EBM__*.pkl` | 3 modelos EBM serializados |
| `docs/html/fase5/m05_mlp_ebm.html` | Informe HTML |

## 🔄 Flujo

```
X_train_prep / y_train  (f5_m01_preparacion)
    ↓ Optuna: MLP 30 trials · EBM 20 trials
    ↓ 5-Fold CV × 6 combinaciones (2 modelos × 3 estrategias)
    ↓ Arquitectura MLP: capas vs AUC
    ↓ Feature importance EBM nativo (InterpretML)
    ↓ Calibración + ROC + PR curve
    ↓ Ajuste umbral óptimo
    ↓ codecarbon
    → results_mlp_ebm.parquet + .json + .pkl + HTML
```

## ➡️ Siguiente

`f5_m06_ensambles.ipynb` — VotingClassifier + StackingClassifier


In [ ]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit,
    cross_validate, learning_curve
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, log_loss,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from interpret.glassbox import ExplainableBoostingClassifier
    EBM_OK = True
except ImportError:
    EBM_OK = False
    print('⚠️  InterpretML no instalado — pip install interpret --break-system-packages')

try:
    from codecarbon import EmissionsTracker
    CODECARBON_OK = True
except ImportError:
    CODECARBON_OK = False
    print('⚠️  codecarbon no instalado')

warnings.filterwarnings('ignore')

# ROOT detection
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina_desde_fichero

# Rutas
RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_RESULTS  = RUTA_MODELADO / 'results'
RUTA_MODELS   = RUTA_MODELADO / 'models'
RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_RESULTS, RUTA_MODELS, RUTA_HTML_F5])

# Constantes
RANDOM_STATE      = 42
N_SPLITS_CV       = 5
N_TRIALS_MLP      = 30   # MLP: espacio medio-grande (capas, neuronas, lr, dropout)
N_TRIALS_EBM      = 20   # EBM: robusto con defaults, Optuna aporta poco
FAMILIA           = 'mlp_ebm'
ESTRATEGIAS       = ['none', 'balanced', 'smote']
MODELOS_M05       = ['MLP', 'EBM']
COLOR             = '#3182ce'
COLOR_MLP         = '#3182ce'
COLOR_EBM         = '#805ad5'
fmt               = formato_numero_es

info_entorno()
print(f'\n📂 ROOT    : {ROOT}')
print(f'📂 RESULTS : {RUTA_RESULTS}')
print(f'📂 MODELS  : {RUTA_MODELS}')
print(f'\n🧠 InterpretML (EBM): {EBM_OK}')
print(f'🌿 codecarbon       : {CODECARBON_OK}')


In [ ]:
# ============================================================================
# CELDA 2: CARGA DE DATOS
# ============================================================================

X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test  = pd.read_parquet(RUTA_MODELADO / 'X_test.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(RUTA_MODELADO / 'y_test.parquet').squeeze()

pipeline_prep = joblib.load(RUTA_MODELADO / 'pipeline_preprocesamiento.pkl')

ruta_train_prep = RUTA_MODELADO / 'X_train_prep.parquet'
ruta_test_prep  = RUTA_MODELADO / 'X_test_prep.parquet'

if ruta_train_prep.exists() and ruta_test_prep.exists():
    X_train_prep = pd.read_parquet(ruta_train_prep)
    X_test_prep  = pd.read_parquet(ruta_test_prep)
    print('✅ Preprocesados cargados desde disco')
else:
    print('⏳ Aplicando pipeline (primera vez)...')
    X_train_prep = pd.DataFrame(
        pipeline_prep.transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_prep = pd.DataFrame(
        pipeline_prep.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )
    X_train_prep.to_parquet(ruta_train_prep)
    X_test_prep.to_parquet(ruta_test_prep)
    print('✅ Preprocesados generados y guardados')

FEATURE_NAMES = list(X_train.columns)
N_FEATURES    = len(FEATURE_NAMES)

print('=' * 55)
print('DATOS CARGADOS')
print('=' * 55)
print(f'X_train : {fmt(len(X_train))} × {N_FEATURES}  |  abandono: {y_train.mean()*100:.1f}%')
print(f'X_test  : {fmt(len(X_test))}  × {N_FEATURES}  |  abandono: {y_test.mean()*100:.1f}%')
print(f'Pipeline: {pipeline_prep.__class__.__name__} ✅')
print()
print('⚠️  X_test / y_test INTOCABLES — evaluación final solo en test')


In [ ]:
# ============================================================================
# CELDA 3: FUNCIONES AUXILIARES
# ============================================================================

CV = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)


def construir_pipeline(modelo, estrategia: str):
    """Ensambla modelo con estrategia de balance.
    none     → modelo directo
    balanced → class_weight en el modelo (MLP no lo soporta — usa SMOTE)
    smote    → SMOTE antes del modelo (ImbPipeline)
    Nota EBM: ExplainableBoostingClassifier soporta sample_weight nativamente
    pero no class_weight — la estrategia 'balanced' usa SMOTE para ambos.
    """
    if estrategia == 'smote':
        return ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('model', modelo)
        ])
    return Pipeline([('model', modelo)])


def evaluar_cv(nombre: str, modelo, X, y, estrategia: str) -> dict:
    """5-Fold CV estratificado con métricas completas + tiempo + memoria."""
    pipe = construir_pipeline(modelo, estrategia)
    scoring = {'f1': 'f1', 'roc_auc': 'roc_auc',
               'precision': 'precision', 'recall': 'recall', 'accuracy': 'accuracy'}
    tracemalloc.start()
    t0     = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    t_total = time.time() - t0
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'familia'        : FAMILIA,
        'f1_mean'        : cv_res['test_f1'].mean(),
        'f1_std'         : cv_res['test_f1'].std(),
        'auc_mean'       : cv_res['test_roc_auc'].mean(),
        'auc_std'        : cv_res['test_roc_auc'].std(),
        'precision_mean' : cv_res['test_precision'].mean(),
        'recall_mean'    : cv_res['test_recall'].mean(),
        'accuracy_mean'  : cv_res['test_accuracy'].mean(),
        'tiempo_s'       : round(t_total, 3),
        'memoria_mb'     : round(mem_pico / 1024**2, 2),
    }


def calcular_metricas_test(nombre: str, pipe_fit, X_te, y_te, estrategia: str) -> dict:
    """Métricas completas sobre test para modelo ya entrenado."""
    y_pred  = pipe_fit.predict(X_te)
    y_proba = pipe_fit.predict_proba(X_te)[:, 1]
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'f1_test'        : round(f1_score(y_te, y_pred), 4),
        'auc_test'       : round(roc_auc_score(y_te, y_proba), 4),
        'auc_pr_test'    : round(average_precision_score(y_te, y_proba), 4),
        'precision_test' : round(precision_score(y_te, y_pred), 4),
        'recall_test'    : round(recall_score(y_te, y_pred), 4),
        'accuracy_test'  : round(accuracy_score(y_te, y_pred), 4),
        'kappa_test'     : round(cohen_kappa_score(y_te, y_pred), 4),
        'logloss_test'   : round(log_loss(y_te, y_proba), 4),
    }


print('✅ Funciones auxiliares definidas')
print('   · construir_pipeline  · evaluar_cv  · calcular_metricas_test')


In [ ]:
# ============================================================================
# CELDA 4: HIPERPARÁMETROS ÓPTIMOS — MLP (30 trials) · EBM (20 trials)
# ============================================================================
# Nota técnica MLP (para el tribunal):
#   MLPClassifier es una red neuronal feedforward con capas ocultas totalmente
#   conectadas. El espacio de búsqueda incluye arquitectura (capas y neuronas),
#   función de activación, optimizador, tasa de aprendizaje y regularización L2.
#   MLP no soporta class_weight — el balance se gestiona con SMOTE.
#
# Nota técnica EBM (para el tribunal):
#   ExplainableBoostingMachine (InterpretML) es un GAM aditivo potenciado con
#   boosting por rondas. Cada feature tiene su propia función de forma aprendida.
#   Es interpretable nativamente: f(x) = f1(x1) + f2(x2) + ... + fij(xi, xj).
#   Rendimiento comparable a GBM clásico con interpretabilidad completa.
#
# MLP: 30 trials — espacio grande (arquitectura + hiperparámetros de entrenamiento)
# EBM: 20 trials — robusto con defaults, Optuna aporta mejoras marginales
# Cambiar FORZAR_OPTUNA = True solo la primera vez.
# ============================================================================

FORZAR_OPTUNA = False

PARAMS_OPTUNA = {
    'MLP': {
        'hidden_layer_sizes' : (100, 50),
        'activation'         : 'relu',
        'solver'             : 'adam',
        'alpha'              : 0.0001,
        'learning_rate_init' : 0.001,
        'max_iter'           : 500,
        'early_stopping'     : True,
        'validation_fraction': 0.1,
    },
    'EBM': {
        'max_bins'           : 256,
        'max_interaction_bins': 32,
        'interactions'       : 10,
        'learning_rate'      : 0.01,
        'max_rounds'         : 5000,
        'min_samples_leaf'   : 2,
    },
}
AUC_OPTUNA = {'MLP': None, 'EBM': None}

if not FORZAR_OPTUNA:
    mejores_params = PARAMS_OPTUNA
    print('⚠️  Params iniciales — ejecutar Optuna la primera vez (FORZAR_OPTUNA = True)')
    print('   Tras ejecutar, registrar los params y poner FORZAR_OPTUNA = False')
    for nombre, params in mejores_params.items():
        print(f'   {nombre}: {params}')
else:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objetivo_mlp(trial):
        n_layers  = trial.suggest_int('n_layers', 1, 4)
        n_units   = trial.suggest_int('n_units', 32, 256)
        hidden    = tuple([n_units] * n_layers)
        params = {
            'hidden_layer_sizes' : hidden,
            'activation'         : trial.suggest_categorical('activation', ['relu', 'tanh']),
            'solver'             : 'adam',
            'alpha'              : trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
            'learning_rate_init' : trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'max_iter'           : 500,
            'early_stopping'     : True,
            'validation_fraction': 0.1,
            'random_state'       : RANDOM_STATE,
        }
        pipe = construir_pipeline(MLPClassifier(**params), 'smote')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)['test_score'].mean()

    def objetivo_ebm(trial):
        if not EBM_OK:
            raise ValueError('InterpretML no instalado')
        params = {
            'max_bins'            : trial.suggest_int('max_bins', 64, 512),
            'interactions'        : trial.suggest_int('interactions', 0, 20),
            'learning_rate'       : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_rounds'          : trial.suggest_int('max_rounds', 1000, 8000),
            'min_samples_leaf'    : trial.suggest_int('min_samples_leaf', 1, 10),
        }
        pipe = construir_pipeline(
            ExplainableBoostingClassifier(**params, random_state=RANDOM_STATE), 'none')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)['test_score'].mean()

    objetivos = {'MLP': (objetivo_mlp, N_TRIALS_MLP), 'EBM': (objetivo_ebm, N_TRIALS_EBM)}
    mejores_params = {}
    for nombre, (objetivo, n_trials) in objetivos.items():
        print(f'🔍 Optuna {nombre} ({n_trials} trials)...', end=' ', flush=True)
        t0    = time.time()
        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objetivo, n_trials=n_trials, show_progress_bar=False)
        AUC_OPTUNA[nombre]    = round(study.best_value, 4)
        mejores_params[nombre] = study.best_params
        print(f'AUC={study.best_value:.4f}  ({time.time()-t0:.0f}s)')
        print(f'   Params: {study.best_params}')

    PARAMS_OPTUNA = mejores_params
    print('\n📋 Copia estos params en PARAMS_OPTUNA y pon FORZAR_OPTUNA = False')


In [ ]:
# ============================================================================
# CELDA 5: ENTRENAMIENTO — 6 COMBINACIONES (2 modelos × 3 estrategias)
# ============================================================================
# Notas técnicas:
#   MLP  : no tiene class_weight — 'balanced' usa SMOTE igual que 'smote'
#          (dos estrategias distintas dan el mismo preprocesado pero
#           Optuna puede encontrar diferente hiperparámetro óptimo)
#   EBM  : no tiene class_weight sklearn — 'balanced' usa SMOTE
# Disk-check: carga pkl si existen, reconstruye métricas si hay pkl sin parquet.
# ============================================================================

def construir_modelo(nombre: str, estrategia: str):
    """Instancia modelo con hiperparámetros registrados de Optuna."""
    p = mejores_params[nombre]
    if nombre == 'MLP':
        # Reconstruir hidden_layer_sizes si vino de Optuna como n_layers/n_units
        if 'n_layers' in p:
            hidden = tuple([p['n_units']] * p['n_layers'])
            p = {k: v for k, v in p.items() if k not in ['n_layers', 'n_units']}
            p['hidden_layer_sizes'] = hidden
        return MLPClassifier(**p, random_state=RANDOM_STATE)
    elif nombre == 'EBM':
        if not EBM_OK:
            raise ImportError('InterpretML no instalado — pip install interpret')
        return ExplainableBoostingClassifier(**p, random_state=RANDOM_STATE)


claves_esperadas = [f'{m}__{e}' for m in MODELOS_M05 for e in ESTRATEGIAS]
pkls_en_disco    = [c for c in claves_esperadas if (RUTA_MODELS / f'{c}.pkl').exists()]
parquet_en_disco = (RUTA_RESULTS / 'results_mlp_ebm.parquet').exists()

print(f'📦 Modelos en disco : {len(pkls_en_disco)}/6')
print(f'📊 Parquet resultados: {parquet_en_disco}')

modelos_fit = {c: joblib.load(RUTA_MODELS / f'{c}.pkl') for c in pkls_en_disco}

if parquet_en_disco:
    df_res = pd.read_parquet(RUTA_RESULTS / 'results_mlp_ebm.parquet')
    df_cv  = df_res.sort_values('auc_mean', ascending=False)
    emisiones = None
    print('✅ Resultados cargados desde disco')

elif len(pkls_en_disco) == 6:
    print('\n⏳ Reconstruyendo métricas (modelos en disco, parquet no)...')
    resultados_cv   = []
    resultados_test = []
    for nombre in MODELOS_M05:
        print(f'   {nombre}...', end=' ')
        for estrategia in ESTRATEGIAS:
            clave = f'{nombre}__{estrategia}'
            pipe  = modelos_fit[clave]
            res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                X_train_prep, y_train, estrategia)
            resultados_cv.append(res_cv)
            res_test = calcular_metricas_test(nombre, pipe, X_test_prep, y_test, estrategia)
            resultados_test.append(res_test)
        print('✅')
    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])
    emisiones = None
    print('✅ Métricas reconstruidas')

else:
    print('\n⏳ Entrenando 6 combinaciones...')
    resultados_cv   = []
    resultados_test = []

    if CODECARBON_OK:
        tracker = EmissionsTracker(project_name=f'TFM_f5_m05_{FAMILIA}',
                                   output_dir=str(RUTA_RESULTS), log_level='error')
        tracker.start()

    print('=' * 60)
    print('ENTRENAMIENTO — 2 modelos × 3 estrategias = 6 combinaciones')
    print('=' * 60)
    for nombre in MODELOS_M05:
        print(f'  {nombre}...', end=' ', flush=True)
        for estrategia in ESTRATEGIAS:
            clave  = f'{nombre}__{estrategia}'
            modelo = construir_modelo(nombre, estrategia)
            res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                X_train_prep, y_train, estrategia)
            resultados_cv.append(res_cv)
            pipe_final = construir_pipeline(modelo, estrategia)
            pipe_final.fit(X_train_prep, y_train)
            modelos_fit[clave] = pipe_final
            joblib.dump(pipe_final, RUTA_MODELS / f'{clave}.pkl')
            res_test = calcular_metricas_test(nombre, pipe_final,
                                              X_test_prep, y_test, estrategia)
            resultados_test.append(res_test)
        print('✅')

    if CODECARBON_OK:
        emisiones = tracker.stop()
        print(f'\n🌿 Emisiones CO₂: {emisiones:.6f} kg')
    else:
        emisiones = None

    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])
    print('\n✅ Entrenamiento completado')

MEJOR_MODELO     = df_cv.iloc[0]['modelo']
MEJOR_ESTRATEGIA = df_cv.iloc[0]['estrategia']
mejor_clave      = f'{MEJOR_MODELO}__{MEJOR_ESTRATEGIA}'
mejor_pipe       = modelos_fit[mejor_clave]

print(f'\n🏆 Mejor: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')


In [ ]:
# ============================================================================
# CELDA 6: TABLA DE RESULTADOS
# ============================================================================

cols_cv = ['modelo', 'estrategia', 'auc_mean', 'auc_std',
           'f1_mean', 'precision_mean', 'recall_mean', 'tiempo_s']

print('=' * 70)
print('RESULTADOS 5-Fold CV — ordenado por AUC')
print('=' * 70)
print(df_cv[cols_cv].to_string(index=False, float_format='{:.4f}'.format))

if 'auc_test' in df_res.columns:
    cols_test = ['modelo', 'estrategia', 'auc_test', 'auc_pr_test',
                 'f1_test', 'precision_test', 'recall_test', 'kappa_test']
    print('\n' + '=' * 70)
    print('MÉTRICAS EN TEST (informativas — selección por CV)')
    print('=' * 70)
    print(df_res[cols_test].sort_values('auc_test', ascending=False)
          .to_string(index=False, float_format='{:.4f}'.format))


In [ ]:
# ============================================================================
# CELDA 7: ANÁLISIS DE ARQUITECTURA MLP (capas vs AUC)
# ============================================================================
# MLP es sensible a la arquitectura (nº de capas y neuronas por capa).
# Se evalúan 4 configuraciones representativas para mostrar el trade-off
# complejidad vs rendimiento — justifica la elección de arquitectura.
# ============================================================================

graficos_b64 = {}

arquitecturas = {
    '(50,)'       : (50,),
    '(100,)'      : (100,),
    '(100, 50)'   : (100, 50),
    '(100, 50, 25)': (100, 50, 25),
    '(200, 100)'  : (200, 100),
}

print('Evaluando arquitecturas MLP (3-Fold CV rápido)...', flush=True)
cv_arq = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
resultados_arq = {}
t0 = time.time()

for nombre_arq, hidden in arquitecturas.items():
    mlp_arq = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation=mejores_params['MLP'].get('activation', 'relu'),
        alpha=mejores_params['MLP'].get('alpha', 0.0001),
        learning_rate_init=mejores_params['MLP'].get('learning_rate_init', 0.001),
        max_iter=300, early_stopping=True,
        random_state=RANDOM_STATE
    )
    pipe_arq = ImbPipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', mlp_arq)
    ])
    cv_r = cross_validate(pipe_arq, X_train_prep, y_train,
                          cv=cv_arq, scoring='roc_auc', n_jobs=-1)
    resultados_arq[nombre_arq] = {
        'auc_mean': cv_r['test_score'].mean(),
        'auc_std' : cv_r['test_score'].std(),
        'params'  : sum(hidden),
    }
    print(f'  {nombre_arq:<18} AUC={cv_r["test_score"].mean():.4f} ± {cv_r["test_score"].std():.4f}')

print(f'✅  ({time.time()-t0:.0f}s)')

# Gráfico
nombres_arq = list(resultados_arq.keys())
aucs_arq    = [resultados_arq[n]['auc_mean'] for n in nombres_arq]
stds_arq    = [resultados_arq[n]['auc_std']  for n in nombres_arq]
best_arq    = nombres_arq[int(np.argmax(aucs_arq))]

colores_arq = [COLOR_MLP if n != best_arq else '#e53e3e' for n in nombres_arq]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(nombres_arq, aucs_arq, color=colores_arq, edgecolor='white', yerr=stds_arq,
              capsize=4)
ymin = max(0.5, min(aucs_arq) - 0.02)
ax.set_ylim(ymin, min(1.0, max(aucs_arq) + 0.03))
ax.set_xlabel('Arquitectura (capas ocultas)')
ax.set_ylabel('AUC-ROC (3-Fold CV)')
ax.set_title('MLP — AUC en función de la arquitectura\n'
             'Rojo = mejor arquitectura evaluada', fontsize=11, fontweight='bold')
for bar, val in zip(bars, aucs_arq):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['arquitectura_mlp'] = figura_a_base64(fig)
plt.close(fig)
print(f'Mejor arquitectura evaluada: {best_arq}  (AUC={max(aucs_arq):.4f})')


In [ ]:
# ============================================================================
# CELDA 8: FEATURE IMPORTANCE EBM — InterpretML nativo
# ============================================================================
# EBM expone la importancia global de cada feature directamente desde el modelo:
# es la magnitud media del efecto de cada feature sobre la predicción.
# A diferencia de Random Forest o XGBoost, la importancia EBM es aditiva y
# se puede interpretar como contribución directa al riesgo de abandono.
# ============================================================================

try:
    # Mejor modelo EBM según CV
    mejor_est_ebm = (df_cv[df_cv['modelo'] == 'EBM']
                     .sort_values('auc_mean', ascending=False)
                     .iloc[0]['estrategia'])
    pipe_ebm = modelos_fit[f'EBM__{mejor_est_ebm}']

    # Extraer el EBM del pipeline
    ebm_model = pipe_ebm.named_steps.get('model') or pipe_ebm[-1]

    # Importancias globales
    importancias_ebm = ebm_model.term_importances()
    feature_names_ebm = ebm_model.term_names_

    # Ordenar top 10
    idx_top = np.argsort(importancias_ebm)[-min(10, len(importancias_ebm)):]
    nombres_top = [feature_names_ebm[i] for i in idx_top]
    valores_top = importancias_ebm[idx_top]

    colores_ebm = ['#e53e3e' if v == valores_top.max() else COLOR_EBM
                   for v in valores_top]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(nombres_top, valores_top, color=colores_ebm, height=0.6)
    ax.set_xlabel('Importancia global EBM (magnitud media del efecto)')
    ax.set_title(
        f'EBM — Feature importance global ({mejor_est_ebm})\n'
        f'Rojo = feature más influyente | Interpretación aditiva nativa',
        fontsize=11, fontweight='bold'
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    graficos_b64['importancia_ebm'] = figura_a_base64(fig)
    plt.close(fig)
    print(f'✅ Feature importance EBM calculada ({mejor_est_ebm})')
    print(f'   Top feature: {nombres_top[-1]}  ({valores_top[-1]:.4f})')
except Exception as e:
    print(f'⚠️  No se pudo calcular importancia EBM: {e}')
    graficos_b64['importancia_ebm'] = None


In [ ]:
# ============================================================================
# CELDA 9: CURVA DE APRENDIZAJE
# ============================================================================
# Se usa submuestra estratificada del 30% del train para mantener tiempos
# razonables con MLP (que puede ser lento con datasets grandes).
# ============================================================================

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.70, random_state=RANDOM_STATE)
idx_sub, _ = next(sss.split(X_train_prep, y_train))
X_sub = X_train_prep.iloc[idx_sub]
y_sub = y_train.iloc[idx_sub]

print(f'Submuestra: {fmt(len(X_sub))} filas '
      f'({len(X_sub)/len(X_train_prep)*100:.0f}% del train)  '
      f'abandono={y_sub.mean()*100:.1f}%')
print('Calculando curva de aprendizaje...', end=' ', flush=True)
t0 = time.time()

train_sizes, train_scores, cv_scores = learning_curve(
    mejor_pipe, X_sub, y_sub,
    cv=CV, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
cv_mean    = cv_scores.mean(axis=1)
cv_std     = cv_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color=COLOR, label='Train AUC', linewidth=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color=COLOR)
ax.plot(train_sizes, cv_mean, 'o-', color='#e53e3e', label='CV AUC (5-fold)', linewidth=2)
ax.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.15, color='#e53e3e')
ax.set_xlabel('Tamaño del conjunto de entrenamiento (submuestra 30%)')
ax.set_ylabel('AUC-ROC')
ax.set_title(
    f'Curva de aprendizaje — {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}\n'
    f'Submuestra estratificada 30% del train',
    fontsize=11, fontweight='bold')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['curva_aprendizaje'] = figura_a_base64(fig)
plt.close(fig)

gap = train_mean[-1] - cv_mean[-1]
print(f'✅  ({time.time()-t0:.0f}s)')
print(f'Gap train-CV: {gap:.4f}  → '
      f'{"Posible overfitting" if gap > 0.05 else "Generalización correcta"}')


In [ ]:
# ============================================================================
# CELDA 10: CALIBRACIÓN DE PROBABILIDADES + CURVAS ROC
# ============================================================================
# Una buena calibración es esencial para el uso institucional:
# p=0.7 debe interpretarse como ~70% de riesgo real de abandono.
# MLP tiende a estar mal calibrado (confianza excesiva cerca de 0 y 1).
# EBM suele estar mejor calibrado por su naturaleza aditiva.
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_2 = {
    'MLP': COLOR_MLP,
    'EBM': COLOR_EBM,
}

ax_cal = axes[0]
ax_cal.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Calibración perfecta')

ax_roc = axes[1]
ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1)

for nombre_m in MODELOS_M05:
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    pipe_m = modelos_fit[f'{nombre_m}__{mejor_est}']
    try:
        y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]
    except Exception:
        continue

    color_m = colores_2.get(nombre_m, COLOR)

    # Calibración
    frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
    ax_cal.plot(mean_pred, frac_pos, 'o-', color=color_m,
                label=f'{nombre_m} ({mejor_est})', linewidth=1.8, markersize=5)

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = roc_auc_score(y_test, y_proba)
    ax_roc.plot(fpr, tpr, color=color_m,
                label=f'{nombre_m} AUC={auc_v:.3f}', linewidth=2)

ax_cal.set_xlabel('Probabilidad predicha')
ax_cal.set_ylabel('Fracción positivos reales')
ax_cal.set_title('Calibración de probabilidades\n'
                 'Desviación de la diagonal = mala calibración', fontweight='bold')
ax_cal.legend(fontsize=9)
ax_cal.spines['top'].set_visible(False)
ax_cal.spines['right'].set_visible(False)

ax_roc.set_xlabel('Tasa de falsos positivos')
ax_roc.set_ylabel('Tasa de verdaderos positivos')
ax_roc.set_title('Curvas ROC comparativas', fontweight='bold')
ax_roc.legend(fontsize=9)
ax_roc.spines['top'].set_visible(False)
ax_roc.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['calibracion_roc'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Calibración + ROC generados')


In [ ]:
# ============================================================================
# CELDA 11: PRECISION-RECALL CURVE + F1 POR UMBRAL
# ============================================================================
# Con desbalance 70/30, la curva PR es más informativa que ROC:
# un modelo puede tener AUC-ROC alto pero PR-AUC bajo si falla en la clase
# minoritaria (los abandonos). AUC-PR = área bajo curva Precision-Recall.
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax_pr   = axes[0]
ax_f1   = axes[1]

umbrales_rango = np.arange(0.05, 0.96, 0.01)

for nombre_m in MODELOS_M05:
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    pipe_m = modelos_fit[f'{nombre_m}__{mejor_est}']
    try:
        y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]
    except Exception:
        continue

    color_m = colores_2.get(nombre_m, COLOR)
    auc_pr  = average_precision_score(y_test, y_proba)

    # Curva PR
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ax_pr.plot(rec, prec, color=color_m, linewidth=2,
               label=f'{nombre_m} AUC-PR={auc_pr:.3f}')

    # F1 por umbral
    f1s = [f1_score(y_test, (y_proba >= t).astype(int), zero_division=0)
           for t in umbrales_rango]
    ax_f1.plot(umbrales_rango, f1s, color=color_m, linewidth=2,
               label=f'{nombre_m} F1_max={max(f1s):.3f}')

# Baseline aleatorio en PR
baseline = y_test.mean()
ax_pr.axhline(baseline, color='gray', linestyle='--', linewidth=1,
              label=f'Baseline aleatorio ({baseline:.2f})')

ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title('Curva Precision-Recall\n'
                'Más informativa que ROC con desbalance 70/30', fontweight='bold')
ax_pr.legend(fontsize=9)
ax_pr.spines['top'].set_visible(False)
ax_pr.spines['right'].set_visible(False)

ax_f1.set_xlabel('Umbral de decisión')
ax_f1.set_ylabel('F1-score')
ax_f1.set_title('F1 en función del umbral\n'
                'Permite elegir umbral operativo óptimo', fontweight='bold')
ax_f1.axvline(0.5, color='gray', linestyle='--', linewidth=1, label='Umbral=0.5')
ax_f1.legend(fontsize=9)
ax_f1.spines['top'].set_visible(False)
ax_f1.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['pr_f1_umbral'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Curva PR + F1 por umbral generados')


In [ ]:
# ============================================================================
# CELDA 12: MATRIZ DE CONFUSIÓN + COMPARATIVA AUC
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

y_pred_mejor = mejor_pipe.predict(X_test_prep)
cm = confusion_matrix(y_test, y_pred_mejor)
ax_cm = axes[0]
im = ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax_cm)
clases = ['Continúa (0)', 'Abandona (1)']
ax_cm.set_xticks([0, 1]); ax_cm.set_xticklabels(clases, rotation=30, ha='right')
ax_cm.set_yticks([0, 1]); ax_cm.set_yticklabels(clases)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                   color='white' if cm[i,j] > thresh else 'black',
                   fontsize=13, fontweight='bold')
ax_cm.set_title(f'Matriz de confusión\n{MEJOR_MODELO} + {MEJOR_ESTRATEGIA} (test)',
                fontweight='bold')
ax_cm.set_ylabel('Etiqueta real')
ax_cm.set_xlabel('Etiqueta predicha')

# Comparativa AUC MLP vs EBM
ax_bar = axes[1]
aucs_cv     = [df_cv[df_cv['modelo']==m]['auc_mean'].max() for m in MODELOS_M05]
colores_bar = [colores_2.get(m, COLOR) for m in MODELOS_M05]
bars = ax_bar.bar(MODELOS_M05, aucs_cv, color=colores_bar, edgecolor='white', width=0.4)
ymin = max(0.5, min(aucs_cv) - 0.02)
ax_bar.set_ylim(ymin, min(1.0, max(aucs_cv) + 0.02))
ax_bar.set_ylabel('AUC-ROC (mejor estrategia, CV)')
ax_bar.set_title('MLP vs EBM — AUC comparativo\n'
                 'Rendimiento vs interpretabilidad', fontweight='bold')
for bar, val in zip(bars, aucs_cv):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['cm_comparativa'] = figura_a_base64(fig)
plt.close(fig)

print('✅ Matriz de confusión + comparativa AUC generados')
print()
print('Classification report — mejor modelo en test:')
print(classification_report(y_test, y_pred_mejor, target_names=['Continúa', 'Abandona']))


In [ ]:
# ============================================================================
# CELDA 13: AJUSTE DE UMBRAL ÓPTIMO
# ============================================================================
# Coste asimétrico del abandono universitario:
#   FN (no detecta abandono real): mayor coste — alumno sin intervención
#   FP (alerta falsa): menor coste — intervención innecesaria
# Se busca umbral F1-óptimo y umbral que garantiza Recall >= 0.75.
# ============================================================================

RECALL_MINIMO   = 0.75
umbrales_result = []
umbrales_rango  = np.arange(0.10, 0.91, 0.01)

print('=' * 60)
print('AJUSTE DE UMBRAL — búsqueda sobre test set')
print('=' * 60)

for nombre in MODELOS_M05:
    mejor_est = (df_cv[df_cv['modelo'] == nombre]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    pipe_m = modelos_fit[f'{nombre}__{mejor_est}']
    y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]

    f1s    = [f1_score(y_test, (y_proba >= t).astype(int), zero_division=0)
              for t in umbrales_rango]
    idx_f1 = int(np.argmax(f1s))
    u_f1   = round(float(umbrales_rango[idx_f1]), 2)
    f1_opt = round(float(f1s[idx_f1]), 4)

    recalls  = [recall_score(y_test, (y_proba >= t).astype(int), zero_division=0)
                for t in umbrales_rango]
    idx_rec  = next((j for j, r in enumerate(recalls) if r >= RECALL_MINIMO), None)
    u_rec    = round(float(umbrales_rango[idx_rec]), 2) if idx_rec is not None else None
    rec_rec  = round(float(recalls[idx_rec]), 4)        if idx_rec is not None else None
    pre_rec  = round(float(precision_score(
                   y_test, (y_proba >= umbrales_rango[idx_rec]).astype(int),
                   zero_division=0)), 4)                if idx_rec is not None else None

    y05      = (y_proba >= 0.50).astype(int)
    f1_05    = round(float(f1_score(y_test, y05, zero_division=0)), 4)
    rec_05   = round(float(recall_score(y_test, y05, zero_division=0)), 4)
    ganancia = round(f1_opt - f1_05, 4)

    umbrales_result.append(dict(
        modelo=nombre, estrategia=mejor_est,
        umbral_f1=u_f1, f1_umbral_opt=f1_opt, f1_umbral_05=f1_05,
        ganancia_f1=ganancia, umbral_recall=u_rec,
        recall_garantizado=rec_rec, precision_recall=pre_rec,
        recall_umbral_05=rec_05,
    ))

    print(f'  {nombre} ({mejor_est})')
    print(f'    Umbral 0.50     → F1={f1_05}  Recall={rec_05}')
    print(f'    Umbral ópt. F1 → {u_f1}  F1={f1_opt}  (+{ganancia})')
    if u_rec is not None:
        print(f'    Umbral Recall≥{RECALL_MINIMO} → {u_rec}  Recall={rec_rec}  Prec={pre_rec}')
    print()

df_umbrales = pd.DataFrame(umbrales_result)
print('✅ Análisis de umbral completado')


In [ ]:
# ============================================================================
# CELDA 14: SERIALIZACIÓN DE RESULTADOS
# ============================================================================

if not (RUTA_RESULTS / 'results_mlp_ebm.parquet').exists():
    df_res.to_parquet(RUTA_RESULTS / 'results_mlp_ebm.parquet', index=False)
    print('✅ results_mlp_ebm.parquet guardado')
else:
    print('✅ results_mlp_ebm.parquet ya existía')

df_res.to_excel(RUTA_RESULTS / 'results_mlp_ebm.xlsx', index=False)
print('✅ results_mlp_ebm.xlsx guardado')

meta = {
    'fecha'            : datetime.now().isoformat(),
    'familia'          : FAMILIA,
    'modelos'          : MODELOS_M05,
    'estrategias'      : ESTRATEGIAS,
    'n_combinaciones'  : 6,
    'n_trials_mlp'     : N_TRIALS_MLP,
    'n_trials_ebm'     : N_TRIALS_EBM,
    'cv_folds'         : N_SPLITS_CV,
    'random_state'     : RANDOM_STATE,
    'mejor_modelo'     : MEJOR_MODELO,
    'mejor_estrategia' : MEJOR_ESTRATEGIA,
    'mejor_auc_cv'     : round(float(df_cv.iloc[0]['auc_mean']), 4),
    'mejor_f1_cv'      : round(float(df_cv.iloc[0]['f1_mean']), 4),
    'mejores_params'   : PARAMS_OPTUNA,
    'auc_optuna'       : AUC_OPTUNA,
    'emisiones_co2_kg' : emisiones,
}
with open(RUTA_RESULTS / 'results_mlp_ebm.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print('✅ results_mlp_ebm.json guardado')
print(f'✅ {len(modelos_fit)} modelos .pkl en {RUTA_MODELS}')


In [ ]:
# ============================================================================
# CELDA 15: GENERAR HTML
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_F5 / 'm05_mlp_ebm.html'

# KPIs
kpis = [
    {'valor': '2',                                  'titulo': 'Modelos'},
    {'valor': '3',                                  'titulo': 'Estrategias'},
    {'valor': '6',                                  'titulo': 'Combinaciones'},
    {'valor': f'{N_TRIALS_MLP}/{N_TRIALS_EBM}',     'titulo': 'Trials MLP/EBM'},
    {'valor': f'{df_cv.iloc[0]["auc_mean"]:.3f}',   'titulo': 'Mejor AUC CV'},
    {'valor': f'{df_cv.iloc[0]["f1_mean"]:.3f}',    'titulo': 'Mejor F1 CV'},
]
kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
    f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
    f'padding:18px 28px;text-align:center;min-width:120px;">'
    f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{k["valor"]}</div>'
    f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{k["titulo"]}</div>'
    f'</div>'
    for k in kpis
) + '</div>'

# Texto dinámico
segundo   = df_cv.iloc[1]
diff_auc  = df_cv.iloc[0]['auc_mean'] - segundo['auc_mean']
diff_f1   = df_cv.iloc[0]['f1_mean']  - segundo['f1_mean']

texto_dinamico = (
    f'<p>El análisis de <strong>6 combinaciones</strong> '
    f'(2 modelos × 3 estrategias) revela que '
    f'<strong>{MEJOR_MODELO}</strong> con estrategia <strong>{MEJOR_ESTRATEGIA}</strong> '
    f'alcanza el mejor AUC CV: '
    f'<strong>{df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}</strong> '
    f'(F1={df_cv.iloc[0]["f1_mean"]:.4f}).</p>'
    f'<p>La diferencia frente al segundo clasificador ({segundo["modelo"]} + {segundo["estrategia"]}) '
    f'es de <strong>{diff_auc:.4f} puntos de AUC</strong> y <strong>{diff_f1:.4f} de F1</strong>.</p>'
    f'<p><strong>MLP vs EBM:</strong> ambos modelos ofrecen capacidad no lineal sobre datos tabulares. '
    f'MLP es una red neuronal feedforward —potente pero opaca—; '
    f'EBM es un GAM aditivo con boosting —rendimiento comparable con interpretabilidad nativa—. '
    f'Si la diferencia de AUC es &lt;0.01, EBM es preferible por su valor explicativo '
    f'ante el tribunal y los stakeholders institucionales.</p>'
)

# Tabla pivotada
filas_pivot = ''
for modelo in MODELOS_M05:
    sub      = df_cv[df_cv['modelo'] == modelo].sort_values('auc_mean', ascending=False).iloc[0]
    es_mejor = modelo == MEJOR_MODELO
    bg       = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    estrella = ' 🏆' if es_mejor else ''
    auc_opt  = AUC_OPTUNA.get(modelo)
    auc_opt_str = f'{auc_opt:.4f}' if auc_opt else '—'
    filas_pivot += (
        f'<tr style="{bg}">'
        f'<td>{modelo}{estrella}</td>'
        f'<td>{sub["estrategia"]}</td>'
        f'<td>{sub["auc_mean"]:.4f} ± {sub["auc_std"]:.4f}</td>'
        f'<td>{sub["f1_mean"]:.4f}</td>'
        f'<td>{sub["precision_mean"]:.4f}</td>'
        f'<td>{sub["recall_mean"]:.4f}</td>'
        f'<td>{auc_opt_str}</td>'
        f'<td><code>{json.dumps(PARAMS_OPTUNA.get(modelo, {}))}</code></td>'
        f'</tr>'
    )
tabla_pivot = (
    '<p>Una fila por modelo con su mejor estrategia. 🏆 = mejor combinación global.</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Mejor estrategia</th>'
    '<th>AUC CV (mean±std)</th><th>F1 CV</th>'
    '<th>Precisión</th><th>Recall</th>'
    '<th>AUC Optuna</th><th>Hiperparámetros</th>'
    f'</tr></thead><tbody>{filas_pivot}</tbody></table>'
)

# Tabla completa CV
filas_cv = ''
for _, r in df_cv.iterrows():
    es_mejor = r['modelo'] == MEJOR_MODELO and r['estrategia'] == MEJOR_ESTRATEGIA
    bg = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    filas_cv += (
        f'<tr style="{bg}">'
        f'<td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} ± {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td>'
        f'<td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td>'
        f'<td>{r["tiempo_s"]:.1f}s</td></tr>'
    )
tabla_cv = (
    f'<p>Ordenado por AUC-ROC. Destacado = mejor combinación: '
    f'<strong>{MEJOR_MODELO} + {MEJOR_ESTRATEGIA}</strong>.</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Estrategia</th><th>AUC (mean±std)</th>'
    '<th>F1</th><th>Precisión</th><th>Recall</th><th>Tiempo</th>'
    f'</tr></thead><tbody>{filas_cv}</tbody></table>'
)

# CO2
co2_html = (
    f'<p>🌿 Huella CO₂: <strong>{emisiones:.6f} kg CO₂eq</strong></p>'
    if emisiones
    else '<p>ℹ️ Modelos cargados desde disco — sin nuevo entrenamiento.</p>'
)

def img_tag(key, caption=''):
    b64 = graficos_b64.get(key)
    if not b64:
        return '<p><em>Gráfico no disponible</em></p>'
    cap = f'<p style="font-size:0.85rem;color:#666;margin-top:4px;">{caption}</p>' if caption else ''
    return f'<img src="data:image/png;base64,{b64}" style="max-width:100%;border-radius:8px;">{cap}'

secciones = (
    '<section class="seccion"><h2>Resumen</h2>' + kpis_html + '</section>'
    '<section class="seccion"><h2>Conclusiones del análisis</h2>' + texto_dinamico + '</section>'
    '<section class="seccion"><h2>Comparativa por modelo — mejor estrategia</h2>' + tabla_pivot + '</section>'
    '<section class="seccion"><h2>Resultados completos 5-Fold CV — 6 combinaciones</h2>' + tabla_cv + '</section>'
    f'<section class="seccion"><h2>Hiperparámetros — Optuna ({N_TRIALS_MLP} trials MLP · {N_TRIALS_EBM} trials EBM)</h2>'
    f'<p>MLP: espacio de búsqueda medio-grande (arquitectura + lr + regularización). '
    f'EBM: robusto con defaults, Optuna aporta mejora marginal.</p>'
    f'<pre style="background:#f7fafc;padding:12px;border-radius:8px;font-size:0.85rem;">'
    f'MLP : {json.dumps(PARAMS_OPTUNA.get("MLP", {}), indent=2)}\n'
    f'EBM : {json.dumps(PARAMS_OPTUNA.get("EBM", {}), indent=2)}'
    f'</pre></section>'
    '<section class="seccion"><h2>Arquitectura MLP — capas vs AUC</h2>'
    '<p>Trade-off complejidad vs rendimiento. Añadir más capas mejora marginalmente '
    'el AUC pero aumenta el riesgo de overfitting y el tiempo de entrenamiento.</p>'
    + img_tag('arquitectura_mlp') + '</section>'
    '<section class="seccion"><h2>EBM — Feature importance global (InterpretML)</h2>'
    '<p>Importancia aditiva nativa: magnitud media del efecto de cada feature '
    'sobre la predicción. Directamente interpretable — no requiere SHAP.</p>'
    + img_tag('importancia_ebm') + '</section>'
    f'<section class="seccion"><h2>Curva de aprendizaje — {MEJOR_MODELO}</h2>'
    + img_tag('curva_aprendizaje',
              'Submuestra 30% del train. Convergencia train/CV indica buena generalización.') + '</section>'
    '<section class="seccion"><h2>Calibración de probabilidades y curvas ROC</h2>'
    '<p>MLP tiende a estar mal calibrado (confianza extrema). '
    'EBM suele estar mejor calibrado por su naturaleza aditiva.</p>'
    + img_tag('calibracion_roc') + '</section>'
    '<section class="seccion"><h2>Curva Precision-Recall + F1 por umbral</h2>'
    '<p>Con desbalance 70/30, AUC-PR es más informativo que AUC-ROC. '
    'La curva F1 por umbral permite seleccionar el umbral operativo óptimo.</p>'
    + img_tag('pr_f1_umbral') + '</section>'
    '<section class="seccion"><h2>Matriz de confusión y comparativa AUC</h2>'
    + img_tag('cm_comparativa') + '</section>'
    '<section class="seccion"><h2>Sostenibilidad computacional</h2>' + co2_html + '</section>'
)

render_pagina_desde_fichero(
    'f5_m05_mlp_ebm.ipynb',
    secciones,
    RUTA_HTML_SALIDA,
    carpeta_notebook='fase5_modelado'
)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')


In [ ]:
# ============================================================================
# CELDA 16: RESUMEN FINAL
# ============================================================================

print('=' * 60)
print('RESUMEN FINAL — f5_m05_mlp_ebm')
print('=' * 60)
print(f'Familia     : MLP + EBM (red neuronal + modelo interpretable)')
print(f'Modelos     : MLPClassifier · ExplainableBoostingMachine')
print(f'Estrategias : none · balanced · smote  (6 combinaciones)')
print(f'Optuna      : {N_TRIALS_MLP} trials MLP · {N_TRIALS_EBM} trials EBM')
print(f'CV folds    : {N_SPLITS_CV}')
print()
print(f'🏆 Mejor: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')
print()
print('📁 Archivos:')
print('   data/05_modelado/results/results_mlp_ebm.parquet')
print('   data/05_modelado/results/results_mlp_ebm.xlsx')
print('   data/05_modelado/results/results_mlp_ebm.json')
print('   data/05_modelado/models/  (6 × .pkl)')
print('   docs/html/fase5/m05_mlp_ebm.html')
print()
print('➡️  Siguiente: f5_m06_ensambles.ipynb')
